# Business Entity Resolution Pipeline

**Challenge:** ML Hackathon 2026  
**Metric:** F0.5 (macro-averaged, precision-heavy)  
**Implementation:** Single notebook pipeline  

---

## Section 0: Setup & Constants

In [1]:
# All imports
import pandas as pd
import numpy as np
import re
import gc
import os
import json
import joblib
import unicodedata
from collections import defaultdict
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try importing optional libs
try:
    import jellyfish
except ImportError:
    print("Run: pip install jellyfish")
    jellyfish = None
try:
    from rapidfuzz.distance import Levenshtein as RapidLevenshtein
except ImportError:
    print("Run: pip install rapidfuzz")
    RapidLevenshtein = None
try:
    import lightgbm as lgb
except ImportError:
    print("Run: pip install lightgbm")
    lgb = None

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix, save_npz, load_npz
from scipy.spatial.distance import cosine as scipy_cosine

# ── Checkpoint helpers ────────────────────────────────────────────────────────
# Every expensive stage saves its output to disk immediately after completion.
# On kernel restart, each stage checks for its checkpoint file and skips
# recomputation if it already exists. This lets you continue from exactly
# where you left off without re-running anything.

def ckpt_exists(path):
    """Return True if a checkpoint file already exists on disk."""
    return Path(str(path)).exists()

def save_json(obj, path):
    """Save a dict/list to JSON on disk."""
    with open(str(path), 'w', encoding='utf-8') as f:
        json.dump(obj, f)
    print(f"  [SAVED] {path}")

def load_json(path):
    """Load a dict/list from JSON on disk."""
    with open(str(path), encoding='utf-8') as f:
        return json.load(f)

def save_df(df, path):
    """Save a DataFrame as parquet (fast, preserves dtypes)."""
    df.to_parquet(str(path), index=False)
    print(f"  [SAVED] {path}")

def load_df(path):
    """Load a DataFrame from parquet."""
    return pd.read_parquet(str(path))

def ckpt_status():
    """Print status of all checkpoint files so you know where to resume."""
    from pathlib import Path
    BASE_DIR = Path("/run/media/devaprasath/FedoraMaxima/ML Hackathon/Aws_ml_dataset/student_resource")
    checkpoints = [
        ("Section 1 — Normalized S1",         BASE_DIR / "models/ckpt_s1_norm.parquet"),
        ("Section 1 — Normalized S2",         BASE_DIR / "models/ckpt_s2_norm.parquet"),
        ("Section 1 — Normalized S3",         BASE_DIR / "models/ckpt_s3_norm.parquet"),
        ("Section 3b — Train candidates",     BASE_DIR / "models/train_candidates.json"),
        ("Section 4b — Name TF-IDF",          BASE_DIR / "models/name_tfidf.pkl"),
        ("Section 4b — Addr TF-IDF",          BASE_DIR / "models/addr_tfidf.pkl"),
        ("Section 5  — LightGBM model",       BASE_DIR / "models/lgbm_classifier.pkl"),
        ("Section 5  — Val split IDs",        BASE_DIR / "models/val_s1_ids.json"),
        ("Section 6  — Optimal threshold",    BASE_DIR / "models/optimal_threshold.json"),
        ("Section 7  — Test candidates",      BASE_DIR / "models/test_candidates.json"),
        ("Section 8  — matching_results.tsv", BASE_DIR / "output/matching_results.tsv"),
        ("Section 8  — candidate_pairs.tsv",  BASE_DIR / "output/candidate_pairs.tsv"),
    ]
    print('\n=== CHECKPOINT STATUS ===')
    for label, path in checkpoints:
        status = '✓ DONE' if Path(str(path)).exists() else '○ PENDING'
        print(f'  {status}  {label}')
    print()

print("Imports complete.")
print("Checkpoint helpers ready. Run ckpt_status() to see pipeline progress.")

Imports complete.
Checkpoint helpers ready. Run ckpt_status() to see pipeline progress.


In [2]:
# Path constants
BASE = Path("/run/media/devaprasath/FedoraMaxima/ML Hackathon/Aws_ml_dataset/student_resource")
TRAIN_S1_PATH = BASE / "dataset/train/train_source1.tsv"
TRAIN_S2_PATH = BASE / "dataset/train/train_source2.tsv"
TRAIN_S3_PATH = BASE / "dataset/train/train_source3.tsv"
TRAIN_GT_PATH = BASE / "dataset/train/train_ground_truth.tsv"
TEST_S1_PATH  = BASE / "dataset/test/test_source1.tsv"
TEST_S2_PATH  = BASE / "dataset/test/test_source2.tsv"
TEST_S3_PATH  = BASE / "dataset/test/test_source3.tsv"
MODELS_DIR    = BASE / "models"
OUTPUT_DIR    = BASE / "output"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

REQUIRED_COLS = ["entity_id", "business_name", "business_address", "country"]
RANDOM_SEED = 42
MAX_CANDIDATES = 500
NEG_SAMPLE_RATIO = 20

FEATURE_NAMES = [
    "name_jaccard", "name_levenshtein", "name_tfidf_cosine", "name_token_overlap",
    "name_char_ngram_sim", "name_length_diff", "name_prefix_match", "name_suffix_match",
    "addr_token_overlap", "addr_levenshtein", "addr_numeric_overlap", "addr_empty_flag",
    "addr_tfidf_cosine", "country_match", "name_addr_product", "max_similarity",
    "is_domain_candidate"
]

# Legal suffix expansions — English + French
LEGAL_SUFFIX_MAP = {
    "ltd": "limited", "llc": "limited liability company",
    "llp": "limited liability partnership", "inc": "incorporated",
    "corp": "corporation", "pvt": "private", "co": "company",
    "plc": "public limited company", "lp": "limited partnership",
    "pc": "professional corporation", "assoc": "associates",
    "intl": "international", "natl": "national",
    # French
    "sarl": "societe a responsabilite limitee",
    "sas": "societe par actions simplifiee",
    "sasu": "societe par actions simplifiee unipersonnelle",
    "eurl": "entreprise unipersonnelle a responsabilite limitee",
    "sci": "societe civile immobiliere",
}

STREET_TYPE_MAP = {
    "st": "street", "rd": "road", "ave": "avenue", "blvd": "boulevard",
    "dr": "drive", "ln": "lane", "ct": "court", "pl": "place",
    "hwy": "highway", "pkwy": "parkway", "sq": "square",
    "rue": "rue", "allee": "allee",
}

STOP_WORDS = frozenset([
    "the","and","of","a","an","in","on","at","for","to","with","by","from",
    "as","is","it","its","be","are","was","were","that","this","or","not",
])

# Canonical legal suffix tokens (removed from blocking keys)
LEGAL_TOKENS = frozenset([
    "limited","company","corporation","incorporated","partnership","private",
    "public","associates","services","international","national","societe",
    "limitee","responsabilite","actions","simplifiee","unipersonnelle",
    "entreprise","civile","immobiliere","llc","llp","ltd","inc","corp",
    "pvt","co","plc","lp","pc","sarl","sas","sasu","eurl","sci",
])

LGBM_PARAMS = {
    "objective": "binary",
    "metric": "binary_logloss",
    "num_leaves": 127,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_child_samples": 20,
    "scale_pos_weight": NEG_SAMPLE_RATIO,
    "n_estimators": 500,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

print("Setup complete. All constants and dictionaries defined.")
print(f"  LEGAL_SUFFIX_MAP: {len(LEGAL_SUFFIX_MAP)} entries")
print(f"  STREET_TYPE_MAP:  {len(STREET_TYPE_MAP)} entries")
print(f"  FEATURE_NAMES:    {len(FEATURE_NAMES)} features")

Setup complete. All constants and dictionaries defined.
  LEGAL_SUFFIX_MAP: 18 entries
  STREET_TYPE_MAP:  13 entries
  FEATURE_NAMES:    17 features


---
## Section 1: Data Ingestion & Validation

In [3]:
def load_source(path, expected_prefix):
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Source file not found: {path}")
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8")
    # Detect wrong separator (single-column result)
    if len(df.columns) == 1:
        raise ValueError(f"File loaded as single column — check sep='\\t': {path}")
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing} in {path}")
    df = df[REQUIRED_COLS].copy()
    # Log prefix warnings
    wrong = df[~df["entity_id"].str.startswith(expected_prefix)]
    if len(wrong) > 0:
        print(f"  WARNING: {len(wrong)} entity_ids in {path} don't start with '{expected_prefix}'")
    return df


def load_ground_truth(path):
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Ground truth not found: {path}")
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8")
    if len(df.columns) == 1:
        raise ValueError(f"Ground truth loaded as single column — check sep='\\t': {path}")
    return df


def validate_training_integrity(df_s1, df_s2, df_s3, df_gt):
    s1_ids = set(df_s1["entity_id"])
    gt_s1_ids = set(df_gt["source1_entity_id"])
    if s1_ids != gt_s1_ids:
        only_in_s1 = s1_ids - gt_s1_ids
        only_in_gt = gt_s1_ids - s1_ids
        raise ValueError(
            f"GT / S1 entity_id mismatch: {len(only_in_s1)} only in S1, "
            f"{len(only_in_gt)} only in GT. Samples: {list(only_in_gt)[:5]}"
        )
    # Check dangling references
    valid_cands = set(df_s2["entity_id"]) | set(df_s3["entity_id"])
    dangling = []
    for ids_str in df_gt["matched_entity_ids"]:
        if ids_str:
            for eid in ids_str.split(","):
                eid = eid.strip()
                if eid and eid not in valid_cands:
                    dangling.append(eid)
    if dangling:
        raise ValueError(
            f"{len(dangling)} dangling GT references. Samples: {dangling[:10]}"
        )
    print("  Integrity check PASSED")


def log_summary(df, name):
    print(f"\n--- {name} ---")
    print(f"  Rows: {len(df):,}")
    for col in df.columns:
        null_count = df[col].isna().sum() + (df[col] == "").sum()
        print(f"  {col}: {null_count:,} null/empty ({null_count/len(df)*100:.2f}%)")
    if "country" in df.columns:
        top_countries = df["country"].value_counts().head(10)
        print(f"  country distribution (top 10):\n{top_countries.to_string()}")


print("Section 1 functions defined.")

Section 1 functions defined.


In [4]:
# --- Load training data ---
print("Loading training data...")
df_s1 = load_source(TRAIN_S1_PATH, "S1-")
df_s2 = load_source(TRAIN_S2_PATH, "S2-")
df_s3 = load_source(TRAIN_S3_PATH, "S3-")
df_gt = load_ground_truth(TRAIN_GT_PATH)

print("Validating training integrity...")
validate_training_integrity(df_s1, df_s2, df_s3, df_gt)

log_summary(df_s1, "train_source1")
log_summary(df_s2, "train_source2")
log_summary(df_s3, "train_source3")
print(f"\nGround truth rows: {len(df_gt):,}")

Loading training data...
Validating training integrity...
  Integrity check PASSED

--- train_source1 ---
  Rows: 2,206,821
  entity_id: 0 null/empty (0.00%)
  business_name: 0 null/empty (0.00%)
  business_address: 0 null/empty (0.00%)
  country: 0 null/empty (0.00%)
  country distribution (top 10):
country
US       1323633
India     883188

--- train_source2 ---
  Rows: 5,034,616
  entity_id: 0 null/empty (0.00%)
  business_name: 0 null/empty (0.00%)
  business_address: 168,967 null/empty (3.36%)
  country: 0 null/empty (0.00%)
  country distribution (top 10):
country
US       3016817
India    2017799

--- train_source3 ---
  Rows: 5,285,603
  entity_id: 0 null/empty (0.00%)
  business_name: 0 null/empty (0.00%)
  business_address: 175,916 null/empty (3.33%)
  country: 0 null/empty (0.00%)
  country distribution (top 10):
country
US       3170056
India    2115547

Ground truth rows: 2,206,821


---
## Section 2: Text Normalization

In [5]:
# Build compiled regex for suffix expansion
def _build_suffix_regex(mapping):
    # Sort by length descending to avoid partial matches
    keys = sorted(mapping.keys(), key=len, reverse=True)
    pattern = r'\b(' + '|'.join(re.escape(k) for k in keys) + r')\b'
    return re.compile(pattern)

_NAME_SUFFIX_RE = _build_suffix_regex(LEGAL_SUFFIX_MAP)
_ADDR_STREET_RE = _build_suffix_regex(STREET_TYPE_MAP)
_PUNCT_RE = re.compile(r'[^\w\s]')  # remove non-alphanumeric-whitespace
_SPACE_RE = re.compile(r'\s+')
_AMPERSAND_RE = re.compile(r'&')
_HYPHEN_RE = re.compile(r'-')
_DOMAIN_RE = re.compile(r'\S+\.(com|net|org|in|co\.in|io|biz|fr)\b', re.IGNORECASE)


def normalize_name(name):
    if not name or pd.isna(name):
        return ""
    # Step 1: lowercase
    s = str(name).lower()
    # Step 2: strip
    s = s.strip()
    # Step 3: punctuation normalize
    s = _AMPERSAND_RE.sub(' and ', s)
    s = _HYPHEN_RE.sub(' ', s)
    # Remove ASCII punctuation only (preserve non-ASCII / unicode)
    s = re.sub(r'[^\w\s]', ' ', s)   # \w matches unicode chars too
    # Step 4: expand legal suffixes (whole-token, word boundary)
    s = _NAME_SUFFIX_RE.sub(lambda m: LEGAL_SUFFIX_MAP[m.group(0)], s)
    # Collapse spaces
    s = _SPACE_RE.sub(' ', s).strip()
    return s


def normalize_address(address):
    if not address or pd.isna(address):
        return ""
    s = str(address).lower().strip()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = _ADDR_STREET_RE.sub(lambda m: STREET_TYPE_MAP[m.group(0)], s)
    s = _SPACE_RE.sub(' ', s).strip()
    return s


def normalize_dataframe(df):
    df = df.copy()
    df["norm_name"]    = df["business_name"].apply(normalize_name)
    df["norm_address"] = df["business_address"].apply(normalize_address)
    df["country"]      = df["country"].str.strip()
    return df


print("Section 2 functions defined.")

Section 2 functions defined.


In [6]:
# --- Normalize training DataFrames (with checkpoint) ---
S1_NORM_CKPT = MODELS_DIR / 'ckpt_s1_norm.parquet'
S2_NORM_CKPT = MODELS_DIR / 'ckpt_s2_norm.parquet'
S3_NORM_CKPT = MODELS_DIR / 'ckpt_s3_norm.parquet'

if ckpt_exists(S1_NORM_CKPT) and ckpt_exists(S2_NORM_CKPT) and ckpt_exists(S3_NORM_CKPT):
    print("[CHECKPOINT] Loading normalized DataFrames from disk (skipping normalization)...")
    df_s1 = load_df(S1_NORM_CKPT)
    df_s2 = load_df(S2_NORM_CKPT)
    df_s3 = load_df(S3_NORM_CKPT)
    print(f"  Loaded: S1={len(df_s1):,}  S2={len(df_s2):,}  S3={len(df_s3):,} rows")
else:
    print("Normalizing training data...")
    df_s1 = normalize_dataframe(df_s1)
    df_s2 = normalize_dataframe(df_s2)
    df_s3 = normalize_dataframe(df_s3)
    save_df(df_s1, S1_NORM_CKPT)
    save_df(df_s2, S2_NORM_CKPT)
    save_df(df_s3, S3_NORM_CKPT)

# Verification samples
print("\nNormalization samples:")
for raw, expected in [
    ("ABC Pvt. Ltd.", "abc private limited"),
    ("Ram & Sons LLC", "ram and sons limited liability company"),
    ("ZNB Club SARL", "znb club societe a responsabilite limitee"),
    ("L.L.C.", "limited liability company"),
]:
    result = normalize_name(raw)
    status = "\u2713" if result == expected else f"\u2717 got '{result}'"
    print(f"  '{raw}' \u2192 '{result}'  {status}")

# Test non-ASCII preservation
non_ascii = "\u0930\u093e\u092e \u092e\u093e\u0930\u094d\u0915\u0947\u091f\u093f\u0902\u0917 \u092a\u094d\u0930\u093e\u0907\u0935\u0947\u091f \u0932\u093f\u092e\u093f\u091f\u0947\u0921"
result = normalize_name(non_ascii)
has_devanagari = any(0x0900 <= ord(c) <= 0x097F for c in result)
print(f"  Non-ASCII preserved: {has_devanagari} (sample: '{result[:30]}...')")
print("Normalization complete.")

[CHECKPOINT] Loading normalized DataFrames from disk (skipping normalization)...
  Loaded: S1=2,206,821  S2=5,034,616  S3=5,285,603 rows

Normalization samples:
  'ABC Pvt. Ltd.' → 'abc private limited'  ✓
  'Ram & Sons LLC' → 'ram and sons limited liability company'  ✓
  'ZNB Club SARL' → 'znb club societe a responsabilite limitee'  ✓
  'L.L.C.' → 'l l c'  ✗ got 'l l c'
  Non-ASCII preserved: True (sample: 'र म म र क ट ग प र इव ट ल म ट ड...')
Normalization complete.


---
## Section 3a: Blocking Key Extraction Functions

In [7]:
def get_name_token_keys(norm_name):
    if not norm_name:
        return []
    tokens = norm_name.split()
    return [t for t in tokens if len(t) >= 2 and t not in STOP_WORDS and t not in LEGAL_TOKENS]


def _is_latin(token):
    return all(ord(c) <= 0x024F for c in token if c.isalpha())


def get_phonetic_key(norm_name):
    if not norm_name:
        return []
    if jellyfish is None:
        return []
    tokens = get_name_token_keys(norm_name)
    for tok in tokens:
        if _is_latin(tok) and len(tok) >= 2:
            try:
                code = jellyfish.metaphone(tok)
                if code:
                    return [code]
            except Exception:
                pass
    return []


def get_addr_token_keys(norm_address):
    if not norm_address:
        return []
    keys = []
    for tok in norm_address.split():
        if re.match(r'^\d+$', tok):
            # Strip leading zeros
            stripped = str(int(tok))
            if stripped:
                keys.append(stripped)
        elif len(tok) >= 3 and tok not in STOP_WORDS:
            keys.append(tok)
    return keys


def get_char_ngram_keys(norm_name, n=3):
    if not norm_name:
        return []
    # Only for non-ASCII names
    if not any(ord(c) > 0x007F for c in norm_name):
        return []
    concat = norm_name.replace(' ', '')
    if len(concat) < n:
        return []
    grams = [concat[i:i+n] for i in range(len(concat)-n+1)]
    return [f"ngram_{hash(g) % 1024}" for g in grams]


def get_domain_name_keys(raw_name):
    if not raw_name:
        return []
    name = raw_name.strip().lstrip('@').lstrip('#')
    if not _DOMAIN_RE.search(name):
        return []
    # Strip TLD
    parts = name.rsplit('.', 1)
    base = parts[0] if len(parts) == 2 else name
    # Split on dots and hyphens
    tokens = re.split(r'[.\-_]', base)
    return [t.lower() for t in tokens if len(t) >= 2]


# --- Verification ---
print("Blocking key function tests:")
print(f"  name_token('abc limited liability company'): {get_name_token_keys('abc limited liability company')}")
print(f"  domain_key('ghaziabadconstructions.com'):    {get_domain_name_keys('ghaziabadconstructions.com')}")
print(f"  domain_key('@cornerstoneinvestments'):        {get_domain_name_keys('@cornerstoneinvestments')}")
devanagari_sample = '\u0930\u093e\u092e \u092e\u093e\u0930\u094d\u0915\u0947\u091f\u093f\u0902\u0917'
print(f"  phonetic(devanagari_sample):                  {get_phonetic_key(devanagari_sample)} (expect [])")
print(f"  addr_token('00701 main street'):             {get_addr_token_keys('00701 main street')}")
ngram_sample = '\u0930\u093e\u092e'
print(f"  char_ngram(devanagari_sample):               {get_char_ngram_keys(ngram_sample)[:3]}")

Blocking key function tests:
  name_token('abc limited liability company'): ['abc', 'liability']
  domain_key('ghaziabadconstructions.com'):    ['ghaziabadconstructions']
  domain_key('@cornerstoneinvestments'):        []
  phonetic(devanagari_sample):                  [] (expect [])
  addr_token('00701 main street'):             ['701', 'main', 'street']
  char_ngram(devanagari_sample):               ['ngram_581']


---
## Section 3b: Inverted Index Construction and Candidate Lookup

In [8]:
def build_inverted_indices(df):
    """Build 5 inverted indices for a source DataFrame."""
    idx = {
        "name_token":  defaultdict(list),
        "phonetic":    defaultdict(list),
        "addr_token":  defaultdict(list),
        "char_ngram":  defaultdict(list),
        "domain_name": defaultdict(list),
    }
    for row in df.itertuples(index=False):
        eid = row.entity_id
        nn  = row.norm_name
        na  = row.norm_address
        raw = row.business_name

        for tok in get_name_token_keys(nn):
            idx["name_token"][tok].append(eid)
        for tok in get_phonetic_key(nn):
            idx["phonetic"][tok].append(eid)
        for tok in get_addr_token_keys(na):
            idx["addr_token"][tok].append(eid)
        for tok in get_char_ngram_keys(nn):
            idx["char_ngram"][tok].append(eid)
        for tok in get_domain_name_keys(raw):
            idx["domain_name"][tok].append(eid)
    return idx


def lookup_candidates_one_source(s1_entity_id, s1_norm_name, s1_norm_address,
                                  s1_country, s1_raw_name, index, cand_country_lookup):
    candidates = set()
    for tok in get_name_token_keys(s1_norm_name):
        candidates.update(index["name_token"].get(tok, []))
    for tok in get_phonetic_key(s1_norm_name):
        candidates.update(index["phonetic"].get(tok, []))
    for tok in get_addr_token_keys(s1_norm_address):
        candidates.update(index["addr_token"].get(tok, []))
    for tok in get_char_ngram_keys(s1_norm_name):
        candidates.update(index["char_ngram"].get(tok, []))
    for tok in get_domain_name_keys(s1_raw_name):
        candidates.update(index["domain_name"].get(tok, []))

    candidates.discard(s1_entity_id)

    # Country filter — skip if either country is empty
    if s1_country:
        s1_c = s1_country.lower().strip()
        filtered = set()
        for eid in candidates:
            cand_c = cand_country_lookup.get(eid, "").lower().strip()
            if not cand_c or not s1_c or cand_c == s1_c:
                filtered.add(eid)
        candidates = filtered

    return list(candidates)


def prune_candidates(s1_norm_name, candidate_ids, cand_name_lookup,
                     tfidf_vectorizer, tfidf_matrix, cand_id_to_idx, max_k=MAX_CANDIDATES):
    if len(candidate_ids) <= max_k:
        return candidate_ids
    # Score by TF-IDF cosine
    s1_vec = tfidf_vectorizer.transform([s1_norm_name])
    scores = []
    for eid in candidate_ids:
        idx_i = cand_id_to_idx.get(eid)
        if idx_i is None or tfidf_matrix is None:
            scores.append((eid, 0.0))
            continue
        cand_vec = tfidf_matrix[idx_i]
        dot = s1_vec.multiply(cand_vec).sum()
        s1_norm_val = np.sqrt(s1_vec.multiply(s1_vec).sum())
        c_norm_val  = np.sqrt(cand_vec.multiply(cand_vec).sum())
        sim = float(dot / (s1_norm_val * c_norm_val + 1e-9))
        scores.append((eid, sim))
    scores.sort(key=lambda x: -x[1])
    return [eid for eid, _ in scores[:max_k]]


def generate_all_candidates(df_s1, df_s2, df_s3, index_s2, index_s3,
                             tfidf_vectorizer, tfidf_s2_matrix, tfidf_s3_matrix,
                             s2_id_to_idx, s3_id_to_idx):
    s2_country = dict(zip(df_s2["entity_id"], df_s2["country"]))
    s3_country = dict(zip(df_s3["entity_id"], df_s3["country"]))
    s2_name    = dict(zip(df_s2["entity_id"], df_s2["norm_name"]))
    s3_name    = dict(zip(df_s3["entity_id"], df_s3["norm_name"]))

    # Build combined id-to-index for pruning
    # We'll use s2 first, then s3 offset by len(s2) in a stacked matrix
    # For simplicity: if > MAX_CANDIDATES, just truncate (TF-IDF scoring requires stacked matrix)
    candidates = {}
    total = len(df_s1)
    for i, row in enumerate(df_s1.itertuples(index=False)):
        eid  = row.entity_id
        nn   = row.norm_name
        na   = row.norm_address
        raw  = row.business_name
        ctry = row.country

        c2 = lookup_candidates_one_source(eid, nn, na, ctry, raw, index_s2, s2_country)
        c3 = lookup_candidates_one_source(eid, nn, na, ctry, raw, index_s3, s3_country)
        merged = list(set(c2) | set(c3))

        if len(merged) > MAX_CANDIDATES:
            # Score using name TF-IDF against s2 candidates only, then s3
            # Prune each source separately and re-merge
            c2_pruned = prune_candidates(nn, c2, s2_name,
                                         tfidf_vectorizer, tfidf_s2_matrix,
                                         s2_id_to_idx, MAX_CANDIDATES // 2)
            c3_pruned = prune_candidates(nn, c3, s3_name,
                                         tfidf_vectorizer, tfidf_s3_matrix,
                                         s3_id_to_idx, MAX_CANDIDATES // 2)
            merged = list(set(c2_pruned) | set(c3_pruned))
            if len(merged) > MAX_CANDIDATES:
                merged = merged[:MAX_CANDIDATES]

        candidates[eid] = merged

        if (i + 1) % 100000 == 0:
            print(f"  Generated candidates for {i+1:,} / {total:,} S1 entities...")

    return candidates


print("Section 3b functions defined.")

Section 3b functions defined.


In [ ]:
# --- Build inverted indices ---
print("Building inverted indices for Source2...")
index_s2 = build_inverted_indices(df_s2)
print(f"  name_token index: {len(index_s2['name_token']):,} unique tokens")
print(f"  addr_token index: {len(index_s2['addr_token']):,} unique tokens")

print("Building inverted indices for Source3...")
index_s3 = build_inverted_indices(df_s3)
print(f"  name_token index: {len(index_s3['name_token']):,} unique tokens")

print("Building TF-IDF vectorizer for blocking pruning...")
name_corpus = pd.concat([df_s1["norm_name"], df_s2["norm_name"], df_s3["norm_name"]], ignore_index=True)
name_tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=100000)
name_tfidf.fit(name_corpus.fillna(""))
print(f"  Vocabulary size: {len(name_tfidf.vocabulary_):,}")

# Precompute TF-IDF matrices for S2 and S3
tfidf_s2_matrix = name_tfidf.transform(df_s2["norm_name"].fillna(""))
tfidf_s3_matrix = name_tfidf.transform(df_s3["norm_name"].fillna(""))
s2_id_to_idx = {eid: i for i, eid in enumerate(df_s2["entity_id"])}
s3_id_to_idx = {eid: i for i, eid in enumerate(df_s3["entity_id"])}

print("Generating candidates for training S1 entities...")
print("(This may take 10-20 minutes for large datasets)")
train_candidates = generate_all_candidates(
    df_s1, df_s2, df_s3, index_s2, index_s3,
    name_tfidf, tfidf_s2_matrix, tfidf_s3_matrix,
    s2_id_to_idx, s3_id_to_idx
)

total_pairs = sum(len(v) for v in train_candidates.values())
zero_cand = sum(1 for v in train_candidates.values() if len(v) == 0)
print(f"\nCandidate generation complete:")
print(f"  Total candidate pairs: {total_pairs:,}")
print(f"  Average per S1 entity: {total_pairs/len(df_s1):.1f}")
print(f"  Zero-candidate entities: {zero_cand:,}")
print(f"  Max candidates (any entity): {max(len(v) for v in train_candidates.values())}")

# --- Persist candidates to disk ---
CANDIDATES_PATH = MODELS_DIR / 'train_candidates.json'
print(f"\nSaving candidates to {CANDIDATES_PATH} ...")
with open(str(CANDIDATES_PATH), 'w') as f:
    json.dump(train_candidates, f)
print("  [SAVED] train_candidates.json")


Building inverted indices for Source2...
  name_token index: 810,182 unique tokens
  addr_token index: 583,309 unique tokens
Building inverted indices for Source3...
  name_token index: 862,681 unique tokens
Building TF-IDF vectorizer for blocking pruning...
  Vocabulary size: 100,000
Generating candidates for training S1 entities...
(This may take 10-20 minutes for large datasets)


In [ ]:
# ── OPTIONAL RELOAD CELL ─────────────────────────────────────────────────────
# If the kernel restarted after candidate generation, run this cell instead of
# re-running the full blocking stage (saves 10-20 minutes).
# Requires: Sections 0, 1, 2 already executed in this session.
#
# import json
# CANDIDATES_PATH = MODELS_DIR / 'train_candidates.json'
# print(f'Loading candidates from {CANDIDATES_PATH} ...')
# with open(str(CANDIDATES_PATH)) as f:
#     train_candidates = json.load(f)
# total_pairs = sum(len(v) for v in train_candidates.values())
# print(f'  Loaded {total_pairs:,} candidate pairs for {len(train_candidates):,} S1 entities')
print('(Reload cell — uncomment above lines if kernel was restarted after blocking)')

---
## Section 3c: Blocking Recall Evaluation

In [ ]:
# Parse ground truth into dict and set
gt_dict = {}
for _, row in df_gt.iterrows():
    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]
    if matched:
        gt_dict[s1_id] = set(matched.split(","))
    else:
        gt_dict[s1_id] = set()

gt_set = set()
for s1_id, cands in gt_dict.items():
    for cid in cands:
        gt_set.add((s1_id, cid.strip()))

# Compute recall ceiling on full training set
print("Computing blocking recall ceiling...")
hit = 0
total_true = 0
for s1_id, true_matches in gt_dict.items():
    if not true_matches:
        continue
    cands = set(train_candidates.get(s1_id, []))
    for tm in true_matches:
        total_true += 1
        if tm.strip() in cands:
            hit += 1

recall_ceiling = hit / total_true if total_true > 0 else 0.0
print(f"\nBlocking Recall Ceiling: {recall_ceiling:.4f} ({recall_ceiling*100:.2f}%)")
print(f"True pairs:     {total_true:,}")
print(f"Recovered:      {hit:,}")
print(f"Missed:         {total_true-hit:,}")

if recall_ceiling < 0.90:
    print(f"\nWARNING: Recall ceiling {recall_ceiling:.4f} < 0.90 target!")
    print("Consider adding a TF-IDF approximate retrieval key.")
else:
    print(f"\nRecall ceiling >= 0.90 — PASSED")

---
## Section 4a: Feature Extraction Core Functions

In [ ]:
def feature_normalize(text):
    if not text or pd.isna(text):
        return ""
    s = str(text).lower()
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def compute_jaccard(tokens_a, tokens_b):
    if not tokens_a and not tokens_b:
        return 0.0
    inter = len(tokens_a & tokens_b)
    union = len(tokens_a | tokens_b)
    return inter / union if union > 0 else 0.0


def compute_levenshtein_sim(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    if RapidLevenshtein is not None:
        try:
            return RapidLevenshtein.normalized_similarity(a, b)
        except Exception:
            pass
    # Fallback pure Python
    la, lb = len(a), len(b)
    if la == 0: return 0.0
    if lb == 0: return 0.0
    d = [[0]*(lb+1) for _ in range(la+1)]
    for i in range(la+1): d[i][0] = i
    for j in range(lb+1): d[0][j] = j
    for i in range(1, la+1):
        for j in range(1, lb+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return 1.0 - d[la][lb] / max(la, lb)


def compute_char_ngram_sim(a, b, n=3):
    def ngrams(s):
        s = s.replace(' ', '')
        return set(s[i:i+n] for i in range(len(s)-n+1)) if len(s) >= n else set()
    ga, gb = ngrams(a), ngrams(b)
    return compute_jaccard(ga, gb)


def extract_numeric_tokens(address):
    if not address:
        return set()
    return {str(int(t)) for t in re.findall(r'\d+', address)}


# --- Unit tests ---
print("Feature function tests:")
assert abs(compute_jaccard({"a","b"}, {"b","c"}) - 1/3) < 1e-6
assert compute_jaccard(set(), set()) == 0.0
assert compute_levenshtein_sim("", "") == 1.0
assert compute_levenshtein_sim("abc", "") == 0.0
assert compute_levenshtein_sim("", "abc") == 0.0
assert extract_numeric_tokens("00701 main st") == {"701"}
assert extract_numeric_tokens("03851 Cobble Ridge Dr") == {"3851"}
print("  All feature unit tests PASSED")

---
## Section 4b: Full Feature Vector Extraction and TF-IDF Setup

In [ ]:
def extract_feature_vector(s1_name, s1_addr, s1_country,
                            cand_name, cand_addr, cand_country, cand_entity_id,
                            s1_name_vec, cand_name_vec, s1_addr_vec, cand_addr_vec):
    # Pre-normalize for features
    fn_s1   = feature_normalize(s1_name)
    fn_cand = feature_normalize(cand_name)
    fa_s1   = feature_normalize(s1_addr)
    fa_cand = feature_normalize(cand_addr)

    toks_s1_n   = set(fn_s1.split())   if fn_s1   else set()
    toks_cand_n = set(fn_cand.split()) if fn_cand else set()
    toks_s1_a   = set(fa_s1.split())   if fa_s1   else set()
    toks_cand_a = set(fa_cand.split()) if fa_cand else set()

    # Name features
    name_jaccard     = compute_jaccard(toks_s1_n, toks_cand_n)
    name_levenshtein = compute_levenshtein_sim(fn_s1, fn_cand)

    # TF-IDF cosine — use precomputed sparse vectors
    def tfidf_cosine(v1, v2):
        if v1 is None or v2 is None:
            return 0.0
        dot = v1.multiply(v2).sum()
        n1  = np.sqrt(v1.multiply(v1).sum())
        n2  = np.sqrt(v2.multiply(v2).sum())
        if n1 == 0 or n2 == 0:
            return 0.0
        return float(dot / (n1 * n2))

    name_tfidf_cos = tfidf_cosine(s1_name_vec, cand_name_vec)

    if toks_s1_n and toks_cand_n:
        name_token_ov = len(toks_s1_n & toks_cand_n) / max(len(toks_s1_n), len(toks_cand_n))
    else:
        name_token_ov = 0.0

    name_cng_sim = compute_char_ngram_sim(fn_s1, fn_cand)

    max_len = max(len(fn_s1), len(fn_cand), 1)
    name_len_diff = abs(len(fn_s1) - len(fn_cand)) / max_len

    s1_toks_list   = fn_s1.split()
    cand_toks_list = fn_cand.split()
    name_prefix = 1 if s1_toks_list and cand_toks_list and s1_toks_list[0] == cand_toks_list[0] else 0
    name_suffix = 1 if s1_toks_list and cand_toks_list and s1_toks_list[-1] == cand_toks_list[-1] else 0

    # Address features
    if toks_s1_a and toks_cand_a:
        addr_tok_ov = len(toks_s1_a & toks_cand_a) / max(len(toks_s1_a), len(toks_cand_a))
    else:
        addr_tok_ov = 0.0

    addr_lev = compute_levenshtein_sim(fa_s1[:200], fa_cand[:200])  # truncate for speed

    nums_s1   = extract_numeric_tokens(fa_s1)
    nums_cand = extract_numeric_tokens(fa_cand)
    if nums_s1 and nums_cand:
        addr_num_ov = len(nums_s1 & nums_cand) / max(len(nums_s1), len(nums_cand))
    else:
        addr_num_ov = 0.0

    addr_empty = 1 if not fa_s1 or not fa_cand else 0
    addr_tfidf_cos = tfidf_cosine(s1_addr_vec, cand_addr_vec)

    # Country
    c_s1   = (s1_country   or "").lower().strip()
    c_cand = (cand_country or "").lower().strip()
    country_match = 1 if c_s1 and c_cand and c_s1 == c_cand else 0

    # Combined
    name_addr_prod = name_jaccard * addr_tok_ov
    max_sim = max(name_jaccard, name_tfidf_cos, name_token_ov)
    is_domain = 1 if _DOMAIN_RE.search(str(cand_entity_id) + " " + str(cand_name)) else 0

    return np.array([
        name_jaccard, name_levenshtein, name_tfidf_cos, name_token_ov,
        name_cng_sim, name_len_diff, name_prefix, name_suffix,
        addr_tok_ov, addr_lev, addr_num_ov, addr_empty, addr_tfidf_cos,
        country_match, name_addr_prod, max_sim, is_domain
    ], dtype=np.float32)


def extract_features_batch(candidate_pairs, s1_lookup, cand_lookup,
                            name_vectorizer, addr_vectorizer, batch_size=4096):
    """Extract features for all candidate pairs in batches."""
    N = len(candidate_pairs)
    result = np.zeros((N, len(FEATURE_NAMES)), dtype=np.float32)

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch = candidate_pairs[start:end]

        s1_ids   = [p[0] for p in batch]
        cand_ids = [p[1] for p in batch]

        # Batch transform TF-IDF
        s1_names   = [s1_lookup.get(eid, {}).get("norm_name",   "") for eid in s1_ids]
        cand_names = [cand_lookup.get(eid, {}).get("norm_name", "") for eid in cand_ids]
        s1_addrs   = [s1_lookup.get(eid, {}).get("norm_address","") for eid in s1_ids]
        cand_addrs = [cand_lookup.get(eid, {}).get("norm_address","") for eid in cand_ids]

        s1_name_vecs   = name_vectorizer.transform(s1_names)
        cand_name_vecs = name_vectorizer.transform(cand_names)
        s1_addr_vecs   = addr_vectorizer.transform(s1_addrs)
        cand_addr_vecs = addr_vectorizer.transform(cand_addrs)

        for i, (s1_id, cand_id) in enumerate(batch):
            s1r   = s1_lookup.get(s1_id,   {})
            candr = cand_lookup.get(cand_id, {})
            vec = extract_feature_vector(
                s1r.get("norm_name",""),   s1r.get("norm_address",""),   s1r.get("country",""),
                candr.get("norm_name",""), candr.get("norm_address",""), candr.get("country",""),
                cand_id,
                s1_name_vecs[i], cand_name_vecs[i],
                s1_addr_vecs[i], cand_addr_vecs[i]
            )
            result[start + i] = vec

        if (start // batch_size) % 50 == 0 and start > 0:
            print(f"  Features: {end:,} / {N:,} pairs processed...")

    return result


print("Section 4b functions defined.")

In [ ]:
# --- TF-IDF vectorizers (with checkpoint) ---
NAME_TFIDF_PATH = MODELS_DIR / 'name_tfidf.pkl'
ADDR_TFIDF_PATH = MODELS_DIR / 'addr_tfidf.pkl'

if ckpt_exists(NAME_TFIDF_PATH) and ckpt_exists(ADDR_TFIDF_PATH):
    print("[CHECKPOINT] Loading TF-IDF vectorizers from disk...")
    name_tfidf = joblib.load(str(NAME_TFIDF_PATH))
    addr_tfidf = joblib.load(str(ADDR_TFIDF_PATH))
    print(f"  name_tfidf vocab: {len(name_tfidf.vocabulary_):,}")
    print(f"  addr_tfidf vocab: {len(addr_tfidf.vocabulary_):,}")
else:
    print("Fitting address TF-IDF vectorizer...")
    addr_corpus = pd.concat([df_s1["norm_address"], df_s2["norm_address"], df_s3["norm_address"]], ignore_index=True)
    addr_tfidf = TfidfVectorizer(analyzer='word', ngram_range=(1,1), max_features=50000)
    addr_tfidf.fit(addr_corpus.fillna(""))
    print(f"  Address vocab size: {len(addr_tfidf.vocabulary_):,}")
    joblib.dump(name_tfidf, str(NAME_TFIDF_PATH))
    joblib.dump(addr_tfidf, str(ADDR_TFIDF_PATH))
    print(f"  [SAVED] Vectorizers to {MODELS_DIR}")

# Build lookup dicts for fast access
print("Building lookup dicts...")
s1_lookup = {
    row.entity_id: {"norm_name": row.norm_name, "norm_address": row.norm_address, "country": row.country}
    for row in df_s1.itertuples(index=False)
}
cand_lookup = {}
for row in df_s2.itertuples(index=False):
    cand_lookup[row.entity_id] = {
        "norm_name": row.norm_name, "norm_address": row.norm_address,
        "country": row.country, "business_name": row.business_name
    }
for row in df_s3.itertuples(index=False):
    cand_lookup[row.entity_id] = {
        "norm_name": row.norm_name, "norm_address": row.norm_address,
        "country": row.country, "business_name": row.business_name
    }
print(f"  S1 lookup: {len(s1_lookup):,} entries")
print(f"  Cand lookup: {len(cand_lookup):,} entries")

# Smoke test on 10k pairs
print("\nSmoke test: extracting features for 10,000 sample pairs...")
sample_pairs = []
for s1_id, cands in list(train_candidates.items())[:2000]:
    for c in cands[:5]:
        sample_pairs.append((s1_id, c))
    if len(sample_pairs) >= 10000:
        break

sample_features = extract_features_batch(
    sample_pairs, s1_lookup, cand_lookup, name_tfidf, addr_tfidf, batch_size=1024
)
assert sample_features.shape == (len(sample_pairs), 17), \
    f"Expected shape ({len(sample_pairs)}, 17), got {sample_features.shape}"
assert not np.any(np.isnan(sample_features)), "NaN found in features!"
assert not np.any(np.isinf(sample_features)), "Inf found in features!"
print(f"  Shape: {sample_features.shape}  dtype: {sample_features.dtype}")
print(f"  Feature means: {sample_features.mean(axis=0).round(3)}")
print("  Smoke test PASSED")

---
## Section 5: Model Training

In [ ]:
# ── Section 5: Model Training (with full checkpoint support) ──────────────────
import lightgbm as lgb

MODEL_PATH      = MODELS_DIR / 'lgbm_classifier.pkl'
VAL_IDS_PATH    = MODELS_DIR / 'val_s1_ids.json'
VAL_SCORES_PATH = MODELS_DIR / 'val_scores.npy'
VAL_PAIRS_PATH  = MODELS_DIR / 'val_pairs.json'

# ── Train/val split ───────────────────────────────────────────────────────────
if ckpt_exists(VAL_IDS_PATH):
    print("[CHECKPOINT] Loading train/val split from disk...")
    val_s1_ids   = load_json(VAL_IDS_PATH)
    all_s1_ids   = list(df_s1['entity_id'])
    val_s1_set   = set(val_s1_ids)
    train_s1_ids = [i for i in all_s1_ids if i not in val_s1_set]
    train_s1_set = set(train_s1_ids)
else:
    all_s1_ids = list(df_s1['entity_id'])
    train_s1_ids, val_s1_ids = train_test_split(
        all_s1_ids, test_size=0.20, random_state=RANDOM_SEED
    )
    train_s1_set = set(train_s1_ids)
    val_s1_set   = set(val_s1_ids)
    save_json(val_s1_ids, VAL_IDS_PATH)

print(f"Train split: {len(train_s1_ids):,} S1 entities")
print(f"Val split:   {len(val_s1_ids):,} S1 entities")

# ── Flat pair lists ───────────────────────────────────────────────────────────
train_pairs = [(s1, c) for s1 in train_s1_ids for c in train_candidates.get(s1, [])]

if ckpt_exists(VAL_PAIRS_PATH):
    print("[CHECKPOINT] Loading val pairs from disk...")
    val_pairs = [tuple(p) for p in load_json(VAL_PAIRS_PATH)]
else:
    val_pairs = [(s1, c) for s1 in val_s1_ids for c in train_candidates.get(s1, [])]
    save_json([list(p) for p in val_pairs], VAL_PAIRS_PATH)

print(f"Train pairs: {len(train_pairs):,}")
print(f"Val pairs:   {len(val_pairs):,}")

# ── GT lookup ─────────────────────────────────────────────────────────────────
gt_set = set()
for s1_id, true_ids in gt_dict.items():
    for cid in true_ids:
        gt_set.add((s1_id, cid))

# ── Negative undersampling ────────────────────────────────────────────────────
def build_training_set(X, y, neg_ratio=NEG_SAMPLE_RATIO, seed=RANDOM_SEED):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos = len(pos_idx)
    n_neg_target = min(n_pos * neg_ratio, len(neg_idx))
    rng = np.random.RandomState(seed)
    neg_sample = rng.choice(neg_idx, size=n_neg_target, replace=False)
    all_idx = np.concatenate([pos_idx, neg_sample])
    rng.shuffle(all_idx)
    return X[all_idx], y[all_idx]


def predict_scores(clf, X, batch_size=65536):
    scores = np.zeros(len(X), dtype=np.float64)
    for start in range(0, len(X), batch_size):
        end = min(start + batch_size, len(X))
        scores[start:end] = clf.predict_proba(X[start:end])[:, 1]
    return scores


# ── Train or load model ───────────────────────────────────────────────────────
if ckpt_exists(MODEL_PATH):
    print("[CHECKPOINT] Loading trained LightGBM model from disk...")
    clf = joblib.load(str(MODEL_PATH))
    print(f"  Model loaded: {MODEL_PATH}")
else:
    print("Extracting training features (this will take a while)...")
    X_train_all = extract_features_batch(
        train_pairs, s1_lookup, cand_lookup, name_tfidf, addr_tfidf, batch_size=4096
    )
    y_train_all = np.array(
        [1 if pair in gt_set else 0 for pair in train_pairs], dtype=np.int8
    )
    print(f"  Positives: {y_train_all.sum():,} ({y_train_all.mean()*100:.2f}%)")

    X_train, y_train = build_training_set(X_train_all, y_train_all)
    print(f"  After sampling: {len(y_train):,} pairs (pos={y_train.sum():,} neg={(y_train==0).sum():,})")
    del X_train_all; gc.collect()

    print("Extracting validation features...")
    X_val = extract_features_batch(
        val_pairs, s1_lookup, cand_lookup, name_tfidf, addr_tfidf, batch_size=4096
    )
    y_val = np.array([1 if pair in gt_set else 0 for pair in val_pairs], dtype=np.int8)

    print("Training LightGBM classifier...")
    clf = lgb.LGBMClassifier(**LGBM_PARAMS)
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)]
    )
    joblib.dump(clf, str(MODEL_PATH))
    print(f"  [SAVED] Model to {MODEL_PATH}")

    # Save val scores for threshold section
    val_scores = predict_scores(clf, X_val)
    np.save(str(VAL_SCORES_PATH), val_scores)
    print(f"  [SAVED] Val scores to {VAL_SCORES_PATH}")

# ── Load val scores if already computed ───────────────────────────────────────
if ckpt_exists(VAL_SCORES_PATH) and 'val_scores' not in dir():
    print("[CHECKPOINT] Loading val scores from disk...")
    X_val = extract_features_batch(
        val_pairs, s1_lookup, cand_lookup, name_tfidf, addr_tfidf, batch_size=4096
    )
    val_scores = np.load(str(VAL_SCORES_PATH))
elif 'val_scores' not in dir():
    if 'X_val' not in dir():
        X_val = extract_features_batch(
            val_pairs, s1_lookup, cand_lookup, name_tfidf, addr_tfidf, batch_size=4096
        )
    val_scores = predict_scores(clf, X_val)
    np.save(str(VAL_SCORES_PATH), val_scores)

# Feature importances
fi = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': clf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print("\nFeature importances (top 10):")
print(fi.head(10).to_string(index=False))
print(f"\nVal score stats: min={val_scores.min():.4f}  max={val_scores.max():.4f}  mean={val_scores.mean():.4f}")
print("\nSection 5 complete.")

---
## Section 6: Threshold Optimization and Validation Evaluation

In [ ]:
def compute_entity_f05(predicted_ids, true_ids):
    """Per-entity F0.5 with all edge cases handled."""
    # Both empty = correct singleton prediction
    if not predicted_ids and not true_ids:
        return 1.0
    # True has matches but we predicted nothing
    if not predicted_ids:
        return 0.0
    # True is empty but we predicted something (false positive)
    if not true_ids:
        return 0.0
    pred = set(predicted_ids)
    true = set(true_ids)
    tp = len(pred & true)
    precision = tp / len(pred)
    recall    = tp / len(true)
    if precision + recall == 0:
        return 0.0
    return (1.25 * precision * recall) / (0.25 * precision + recall)


def compute_macro_f05(s1_ids, predictions, gt):
    """Macro-averaged F0.5 over all s1_ids, including Singletons."""
    scores = []
    for s1_id in s1_ids:
        pred = predictions.get(s1_id, set())
        true = gt.get(s1_id, set())
        scores.append(compute_entity_f05(pred, true))
    return float(np.mean(scores)) if scores else 0.0


def apply_threshold(scores, pairs, threshold):
    """Apply threshold (strictly greater than) and group by S1 entity."""
    matched = defaultdict(list)
    for (s1_id, cand_id), score in zip(pairs, scores):
        if score > threshold:  # strictly greater
            matched[s1_id].append(cand_id)
    return dict(matched)


def sweep_thresholds(val_scores, val_pairs, val_s1_ids, gt,
                     t_min=0.0, t_max=1.0, t_step=0.01):
    """Sweep thresholds, return (optimal_t, results_df)."""
    thresholds = np.arange(t_min, t_max + t_step/2, t_step)
    results = []
    best_f05 = -1.0
    best_t   = 0.5

    for t in thresholds:
        predictions = apply_threshold(val_scores, val_pairs, float(t))
        macro_f05   = compute_macro_f05(val_s1_ids, predictions, gt)

        # Compute precision and recall for reporting
        tp = fp = fn = 0
        for s1_id in val_s1_ids:
            pred = set(predictions.get(s1_id, []))
            true = gt.get(s1_id, set())
            tp += len(pred & true)
            fp += len(pred - true)
            fn += len(true - pred)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        results.append({"threshold": round(float(t), 2), "precision": round(precision, 4),
                         "recall": round(recall, 4), "f05": round(macro_f05, 4)})

        # Tie-break: prefer HIGHER threshold on equal F0.5 (favours precision)
        if macro_f05 > best_f05 or (macro_f05 == best_f05 and float(t) > best_t):
            best_f05 = macro_f05
            best_t   = float(t)

    results_df = pd.DataFrame(results)
    return best_t, results_df


# --- Edge case unit tests ---
assert compute_entity_f05(set(), set())    == 1.0, "correct singleton should be 1.0"
assert compute_entity_f05(set(), {"A"})   == 0.0, "missed non-singleton should be 0.0"
assert compute_entity_f05({"A"}, set())   == 0.0, "false positive on singleton should be 0.0"
assert compute_entity_f05({"A"}, {"A"})  == 1.0, "perfect match should be 1.0"
print("F0.5 unit tests PASSED")

In [ ]:
# ── Section 6: Threshold Optimization (with checkpoint) ───────────────────────
THRESHOLD_PATH      = MODELS_DIR / 'optimal_threshold.json'
THRESHOLD_TABLE_TSV = BASE / 'docs/threshold_sweep.tsv'

if ckpt_exists(THRESHOLD_PATH):
    print("[CHECKPOINT] Loading optimal threshold from disk...")
    thresh_data = load_json(THRESHOLD_PATH)
    optimal_t = thresh_data['optimal_threshold']
    threshold_table = pd.read_csv(str(THRESHOLD_TABLE_TSV), sep='\t') \
        if ckpt_exists(THRESHOLD_TABLE_TSV) else None
    best_row = thresh_data
    print(f"  Optimal threshold: {optimal_t:.2f}")
    print(f"  Val F0.5: {thresh_data['f05']:.4f}  Precision: {thresh_data['precision']:.4f}  Recall: {thresh_data['recall']:.4f}")
else:
    print("Running threshold sweep on validation set...")
    val_gt = {s1_id: gt_dict.get(s1_id, set()) for s1_id in val_s1_ids}

    optimal_t, threshold_table = sweep_thresholds(
        val_scores, val_pairs, val_s1_ids, val_gt
    )

    best_row = threshold_table[threshold_table['threshold'] == round(optimal_t, 2)].iloc[0]
    print(f"\nOptimal threshold: {optimal_t:.2f}")
    print(f"  Precision: {best_row['precision']:.4f}")
    print(f"  Recall:    {best_row['recall']:.4f}")
    print(f"  F0.5:      {best_row['f05']:.4f}")

    # Save threshold to disk
    save_json({
        'optimal_threshold': float(optimal_t),
        'precision': float(best_row['precision']),
        'recall':    float(best_row['recall']),
        'f05':       float(best_row['f05'])
    }, THRESHOLD_PATH)

    # Save threshold table
    os.makedirs(str(BASE / 'docs'), exist_ok=True)
    threshold_table.to_csv(str(THRESHOLD_TABLE_TSV), sep='\t', index=False)
    print(f"  [SAVED] Threshold table to {THRESHOLD_TABLE_TSV}")
    print("\nTop 10 threshold candidates by F0.5:")
    print(threshold_table.nlargest(10, 'f05').to_string(index=False))

    # Optional per-country check
    print("\nPer-country F0.5 at global threshold:")
    for country in ['US', 'India']:
        country_s1_ids = [
            s1_id for s1_id in val_s1_ids
            if s1_lookup.get(s1_id, {}).get('country', '') == country
        ]
        if country_s1_ids:
            preds = apply_threshold(val_scores, val_pairs, optimal_t)
            country_gt = {s: val_gt[s] for s in country_s1_ids if s in val_gt}
            f05 = compute_macro_f05(country_s1_ids, preds, country_gt)
            print(f"  {country}: F0.5 = {f05:.4f} ({len(country_s1_ids):,} entities)")

---
## Section 7: Test Inference Pipeline

In [ ]:
# ── Stage 7.1: Load test data ─────────────────────────────────────────────────
print("=" * 60)
print("STAGE 1/7: Loading test data")
print("=" * 60)
df_ts1 = load_source(TEST_S1_PATH, "S1-")
df_ts2 = load_source(TEST_S2_PATH, "S2-")
df_ts3 = load_source(TEST_S3_PATH, "S3-")
print(f"  test_source1: {len(df_ts1):,} rows")
print(f"  test_source2: {len(df_ts2):,} rows")
print(f"  test_source3: {len(df_ts3):,} rows")

# ── Stage 7.2: Normalize ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STAGE 2/7: Normalizing test data")
print("=" * 60)
df_ts1 = normalize_dataframe(df_ts1)
df_ts2 = normalize_dataframe(df_ts2)
df_ts3 = normalize_dataframe(df_ts3)
print(f"  Normalized {len(df_ts1)+len(df_ts2)+len(df_ts3):,} records")

# ── Stage 7.3: Build test indices ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STAGE 3/7: Building blocking indices for test data")
print("=" * 60)
test_index_s2 = build_inverted_indices(df_ts2)
test_index_s3 = build_inverted_indices(df_ts3)

# Load saved vectorizers (DO NOT refit on test data)
name_tfidf_loaded = joblib.load(MODELS_DIR / "name_tfidf.pkl")
addr_tfidf_loaded = joblib.load(MODELS_DIR / "addr_tfidf.pkl")

# Transform test source TF-IDF matrices
tfidf_ts2_matrix = name_tfidf_loaded.transform(df_ts2["norm_name"].fillna(""))
tfidf_ts3_matrix = name_tfidf_loaded.transform(df_ts3["norm_name"].fillna(""))
ts2_id_to_idx    = {eid: i for i, eid in enumerate(df_ts2["entity_id"])}
ts3_id_to_idx    = {eid: i for i, eid in enumerate(df_ts3["entity_id"])}

# Build test lookup dicts
ts1_lookup = {
    row.entity_id: {"norm_name": row.norm_name, "norm_address": row.norm_address, "country": row.country}
    for row in df_ts1.itertuples(index=False)
}
test_cand_lookup = {}
for row in df_ts2.itertuples(index=False):
    test_cand_lookup[row.entity_id] = {
        "norm_name": row.norm_name, "norm_address": row.norm_address,
        "country": row.country, "business_name": row.business_name
    }
for row in df_ts3.itertuples(index=False):
    test_cand_lookup[row.entity_id] = {
        "norm_name": row.norm_name, "norm_address": row.norm_address,
        "country": row.country, "business_name": row.business_name
    }

# ── Stage 7.4: Generate test candidates (with checkpoint) ────────────────────
print("\n" + "=" * 60)
print("STAGE 3/7 continued: Generating test candidates")
print("=" * 60)

TEST_CANDIDATES_PATH  = MODELS_DIR / 'test_candidates.json'
TEST_SCORES_PATH      = MODELS_DIR / 'test_scores.npy'
TEST_PAIRS_PATH       = MODELS_DIR / 'test_pairs.json'

if ckpt_exists(TEST_CANDIDATES_PATH):
    print("[CHECKPOINT] Loading test candidates from disk (skipping blocking)...")
    test_candidates = load_json(TEST_CANDIDATES_PATH)
    total_test_pairs = sum(len(v) for v in test_candidates.values())
    print(f"  Loaded {total_test_pairs:,} candidate pairs for {len(test_candidates):,} S1 entities")
else:
    test_candidates = generate_all_candidates(
        df_ts1, df_ts2, df_ts3,
        test_index_s2, test_index_s3,
        name_tfidf_loaded, tfidf_ts2_matrix, tfidf_ts3_matrix,
        ts2_id_to_idx, ts3_id_to_idx
    )
    total_test_pairs = sum(len(v) for v in test_candidates.values())
    print(f"  Total test candidate pairs: {total_test_pairs:,}")
    print(f"  Average per S1:             {total_test_pairs/len(df_ts1):.1f}")
    print(f"  Zero-candidate entities:    {sum(1 for v in test_candidates.values() if len(v)==0):,}")
    save_json(test_candidates, TEST_CANDIDATES_PATH)

# ── Stage 7.5: Extract test features (with checkpoint) ───────────────────────
print("\n" + "=" * 60)
print("STAGE 4/7: Extracting test features")
print("=" * 60)

if ckpt_exists(TEST_PAIRS_PATH):
    print("[CHECKPOINT] Loading test pairs from disk...")
    test_pairs_flat = [tuple(p) for p in load_json(TEST_PAIRS_PATH)]
else:
    test_pairs_flat = [(s1, c) for s1 in df_ts1["entity_id"] for c in test_candidates.get(s1, [])]
    save_json([list(p) for p in test_pairs_flat], TEST_PAIRS_PATH)

if ckpt_exists(TEST_SCORES_PATH):
    print(f"[CHECKPOINT] Skipping feature extraction — test scores already saved.")
    print(f"  (Will load scores directly in Stage 5)")
    X_test = None  # will be skipped
else:
    print(f"  Extracting features for {len(test_pairs_flat):,} pairs...")
    X_test = extract_features_batch(
        test_pairs_flat, ts1_lookup, test_cand_lookup,
        name_tfidf_loaded, addr_tfidf_loaded, batch_size=4096
    )
    print(f"  Feature matrix shape: {X_test.shape}")

# ── Stage 7.6: Score with model (with checkpoint) ─────────────────────────────
print("\n" + "=" * 60)
print("STAGE 5/7: Scoring with LightGBM model")
print("=" * 60)

if ckpt_exists(TEST_SCORES_PATH):
    print("[CHECKPOINT] Loading test scores from disk...")
    test_scores = np.load(str(TEST_SCORES_PATH))
    print(f"  Loaded {len(test_scores):,} scores")
else:
    clf_loaded = joblib.load(str(MODEL_PATH))
    test_scores = predict_scores(clf_loaded, X_test, batch_size=65536)
    np.save(str(TEST_SCORES_PATH), test_scores)
    print(f"  Scored {len(test_scores):,} pairs")
    print(f"  [SAVED] Test scores to {TEST_SCORES_PATH}")
    del X_test; gc.collect()

print(f"  Score stats: min={test_scores.min():.4f}  max={test_scores.max():.4f}  mean={test_scores.mean():.4f}")

# ── Stage 7.7: Apply threshold ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STAGE 6/7: Applying threshold")
print("=" * 60)
print(f"  Using optimal threshold: {optimal_t:.2f}")
test_matched = apply_threshold(test_scores, test_pairs_flat, optimal_t)
n_matched    = sum(len(v) for v in test_matched.values())
n_singletons = sum(1 for s1_id in df_ts1["entity_id"] if not test_matched.get(s1_id))
print(f"  Total matches predicted: {n_matched:,}")
print(f"  S1 entities with ≥1 match: {len(test_matched):,}")
print(f"  Predicted singletons: {n_singletons:,}")

# Log stage 7 summary
print("\n" + "=" * 60)
print("INFERENCE PIPELINE LOG")
print("=" * 60)
print(f"  1. Records loaded:       S1={len(df_ts1):,}, S2={len(df_ts2):,}, S3={len(df_ts3):,}")
print(f"  2. Records normalized:   {len(df_ts1)+len(df_ts2)+len(df_ts3):,}")
print(f"  3. Candidates generated: {total_test_pairs:,}")
print(f"  4. Features extracted:   {len(test_pairs_flat):,} pairs")
print(f"  5. Pairs scored:         {len(test_scores):,}")
print(f"  6. Matches accepted:     {n_matched:,} (threshold > {optimal_t:.2f})")
print(f"  7. Output rows:          {len(df_ts1):,} (one per S1 entity)")

---
## Section 8: Output File Generation and Submission Validation

In [ ]:
def write_matching_results(test_s1_ids, matched, path):
    """Write matching_results.tsv — one row per S1 entity."""
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(str(path), "w", encoding="utf-8") as f:
        f.write("source1_entity_id\tmatched_entity_ids\n")
        for s1_id in test_s1_ids:
            cand_ids = matched.get(s1_id, [])
            # Sort for determinism; deduplicate
            unique_sorted = sorted(set(cand_ids))
            ids_str = ",".join(unique_sorted)  # empty string if no matches
            f.write(f"{s1_id}\t{ids_str}\n")
    print(f"  Written: {path}")


def write_candidate_pairs(test_s1_ids, candidates, path):
    """Write candidate_pairs.tsv — one row per S1 entity."""
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(str(path), "w", encoding="utf-8") as f:
        f.write("source1_entity_id\tcandidate_entity_ids\n")
        for s1_id in test_s1_ids:
            cand_ids = candidates.get(s1_id, [])
            unique_sorted = sorted(set(cand_ids))
            ids_str = ",".join(unique_sorted)
            f.write(f"{s1_id}\t{ids_str}\n")
    print(f"  Written: {path}")


def run_validator(matching, candidate, test_dir, log_path, max_retries=3):
    """Run validate_submission.py and save output."""
    import subprocess
    cmd = [
        "python3", str(BASE / "utils/validate_submission.py"),
        "--matching",  str(matching),
        "--candidate", str(candidate),
        "--test-dir",  str(test_dir),
    ]
    for attempt in range(1, max_retries + 1):
        result = subprocess.run(cmd, capture_output=True, text=True)
        output = result.stdout + result.stderr
        os.makedirs(os.path.dirname(str(log_path)), exist_ok=True)
        with open(str(log_path), "w") as f:
            f.write(output)
        if result.returncode == 0:
            print(f"  Validator: PASS (exit 0)")
            print(f"  Log saved: {log_path}")
            return True
        else:
            print(f"  Attempt {attempt}/{max_retries}: FAIL (exit {result.returncode})")
            print(output[:500])
            if attempt == max_retries:
                raise RuntimeError(
                    f"Validator failed after {max_retries} attempts. "
                    f"Check {log_path} for details."
                )
    return False


print("Section 8 functions defined.")

In [ ]:
# ── Stage 8: Write output files ───────────────────────────────────────────────
print("=" * 60)
print("STAGE 7/7: Writing output files")
print("=" * 60)

MATCHING_PATH   = OUTPUT_DIR / "matching_results.tsv"
CANDIDATE_PATH  = OUTPUT_DIR / "candidate_pairs.tsv"
VALIDATOR_LOG   = OUTPUT_DIR / "validation_pass.log"
TEST_S1_IDS     = list(df_ts1["entity_id"])  # preserves original order

write_matching_results(TEST_S1_IDS, test_matched, MATCHING_PATH)
write_candidate_pairs(TEST_S1_IDS,  test_candidates, CANDIDATE_PATH)

# ── Verify row counts ─────────────────────────────────────────────────────────
with open(str(MATCHING_PATH)) as f:
    rows = f.readlines()
assert len(rows) == len(TEST_S1_IDS) + 1, \
    f"Expected {len(TEST_S1_IDS)+1} rows (1 header), got {len(rows)}"

# Verify no 'nan' strings
content = open(str(MATCHING_PATH)).read()
assert 'nan' not in content.lower().split(), \
    "'nan' found in matching_results.tsv!"

# Verify every matched ID appears in candidate_pairs
print("\nVerifying matched ⊆ candidates...")
match_df = pd.read_csv(str(MATCHING_PATH),   sep="\t", dtype=str, keep_default_na=False)
cand_df  = pd.read_csv(str(CANDIDATE_PATH),  sep="\t", dtype=str, keep_default_na=False)
cand_map = dict(zip(cand_df["source1_entity_id"], cand_df["candidate_entity_ids"]))
violation = 0
for _, row in match_df.iterrows():
    if not row["matched_entity_ids"]:
        continue
    matched_set   = set(row["matched_entity_ids"].split(","))
    candidate_str = cand_map.get(row["source1_entity_id"], "")
    candidate_set = set(candidate_str.split(",")) if candidate_str else set()
    if not matched_set.issubset(candidate_set):
        violation += 1
assert violation == 0, f"{violation} entities have matched IDs not in candidates!"
print(f"  All matched IDs are subsets of candidates — PASSED")

# ── Run validator ─────────────────────────────────────────────────────────────
print("\nRunning submission validator...")
run_validator(
    MATCHING_PATH, CANDIDATE_PATH,
    BASE / "dataset/test",
    VALIDATOR_LOG
)

print("\n" + "=" * 60)
print("SUBMISSION READY")
print("=" * 60)
print(f"  output/matching_results.tsv  — {len(TEST_S1_IDS):,} rows")
print(f"  output/candidate_pairs.tsv   — {len(TEST_S1_IDS):,} rows")
print(f"  output/validation_pass.log   — validator output")
print(f"  Optimal threshold: {optimal_t:.2f}")
print(f"  Val F0.5: {best_row['f05']:.4f}  Precision: {best_row['precision']:.4f}  Recall: {best_row['recall']:.4f}")